# Lift dataset → nested lists (no padding, no masks)
Kernel: **Python (robomimic)**. Needs `h5py`, `numpy`, `pandas`.

Every demo keeps its own length. Structure:

```
inputs  = [video_1, video_2, ..., video_N]        # one element per demo
video_i = [time_steps, values]
    time_steps = [1, 2, ..., T_i]                 # T_i differs per demo
    values     = [[19 numbers at step 1], [19 numbers at step 2], ...]   # T_i rows

targets = [video_1, ..., video_N]                 # same structure, 7 numbers per step (the human's actions)

input_keys  = ["eef_x", "eef_y", "eef_z", ...]    # names of the 19 columns (shared by all demos)
target_keys = ["act_dx", "act_dy", ...]           # names of the 7 columns
```

## 1. Settings

In [12]:
import os, json
import numpy as np
import h5py
import pandas as pd

DATASET = os.path.expanduser("low_dim_v15.hdf5")

N_DEMOS   = 200               # how many demos to load
SPLIT     = None              # "train", "valid", "20_percent_train", ... or None for all 200 demos
RANDOM    = False             # True = pick demos at random, False = take the first N
SEED      = 0
TIME_STARTS_AT = 1            # time steps numbered 1, 2, ... (use 0 for 0, 1, 2, ...)

STATE_KEYS = ["robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos", "object"]   # 19 input numbers

COLUMN_NAMES = {
    "robot0_eef_pos":      ["eef_x", "eef_y", "eef_z"],
    "robot0_eef_quat":     ["eef_qx", "eef_qy", "eef_qz", "eef_qw"],
    "robot0_gripper_qpos": ["finger_1", "finger_2"],
    "object":              ["cube_x", "cube_y", "cube_z", "cube_qx", "cube_qy", "cube_qz", "cube_qw",
                            "grip2cube_x", "grip2cube_y", "grip2cube_z"],
    "actions":             ["act_dx", "act_dy", "act_dz", "act_drx", "act_dry", "act_drz", "act_grip"],
}
input_keys  = [n for k in STATE_KEYS for n in COLUMN_NAMES[k]]   # 19 names
target_keys = COLUMN_NAMES["actions"]                             # 7 names
print("dataset found:", os.path.exists(DATASET))

dataset found: True


## 2. Select demos

In [13]:
with h5py.File(DATASET, "r") as f:
    env_name = json.loads(f["data"].attrs["env_args"])["env_name"]
    if SPLIT is None:
        pool = sorted(f["data"].keys(), key=lambda s: int(s.split("_")[1]))
    else:
        pool = [d.decode() for d in f[f"mask/{SPLIT}"][:]]
print(f"task: {env_name} | demos available in '{SPLIT}': {len(pool)}")

rng = np.random.default_rng(SEED)
selected = list(rng.choice(pool, size=min(N_DEMOS, len(pool)), replace=False)) if RANDOM else pool[:N_DEMOS]
print(f"selected {len(selected)} demos:", selected[:10], "..." if len(selected) > 10 else "")

task: Lift | demos available in 'None': 200
selected 200 demos: ['demo_0', 'demo_1', 'demo_2', 'demo_3', 'demo_4', 'demo_5', 'demo_6', 'demo_7', 'demo_8', 'demo_9'] ...


## 3. Build the nested lists

In [14]:
inputs, targets = [], []
with h5py.File(DATASET, "r") as f:
    for d in selected:
        g = f[f"data/{d}"]
        obs  = np.concatenate([g[f"obs/{k}"][:] for k in STATE_KEYS], axis=1)   # (T, 19)
        acts = g["actions"][:]                                                 # (T, 7)
        T = obs.shape[0]
        time_steps = list(range(TIME_STARTS_AT, TIME_STARTS_AT + T))
        inputs.append([time_steps, obs.tolist()])                              # [time steps, T x 19 values]
        targets.append([time_steps, acts.tolist()])                            # [time steps, T x 7 values]

lengths = [len(v[0]) for v in inputs]
print("number of videos:", len(inputs))
print(f"time steps per video: min {min(lengths)}, max {max(lengths)}, total {sum(lengths)}")
print("input keys :", input_keys)
print("target keys:", target_keys)

number of videos: 200
time steps per video: min 36, max 64, total 9666
input keys : ['eef_x', 'eef_y', 'eef_z', 'eef_qx', 'eef_qy', 'eef_qz', 'eef_qw', 'finger_1', 'finger_2', 'cube_x', 'cube_y', 'cube_z', 'cube_qx', 'cube_qy', 'cube_qz', 'cube_qw', 'grip2cube_x', 'grip2cube_y', 'grip2cube_z']
target keys: ['act_dx', 'act_dy', 'act_dz', 'act_drx', 'act_dry', 'act_drz', 'act_grip']


## 4. Check the structure

In [15]:
v = 0                                            # which video to inspect
time_steps, values = inputs[v]
_, actions = targets[v]

print(f"video {v} ({selected[v]})")
print("  time steps:", time_steps[:5], "...", time_steps[-3:], f"({len(time_steps)} steps)")
print("  input values at step 1 :", np.round(values[0], 4).tolist(), f"({len(values[0])} numbers)")
print("  target values at step 1:", np.round(actions[0], 4).tolist(), f"({len(actions[0])} numbers)")

# consistency checks (no padding anywhere: every row of every video is real data)
assert all(len(inp[0]) == len(inp[1]) == len(tgt[1]) for inp, tgt in zip(inputs, targets))
assert all(len(row) == len(input_keys)  for inp in inputs  for row in inp[1])
assert all(len(row) == len(target_keys) for tgt in targets for row in tgt[1])
assert not any(np.isnan(inp[1]).any() for inp in inputs)
print("\nall checks passed: lengths match, 19 inputs / 7 targets per step, no NaN")

video 0 (demo_0)
  time steps: [1, 2, 3, 4, 5] ... [57, 58, 59] (59 steps)
  input values at step 1 : [-0.0905, -0.0152, 1.0118, 0.9972, -0.0072, 0.074, 0.0019, 0.0208, -0.0208, 0.0264, 0.027, 0.8314, 0.0, 0.0, 0.9691, 0.2466, 0.1169, 0.0422, -0.1804] (19 numbers)
  target values at step 1: [-0.0, 0.0, 0.0, 0.0038, 0.1482, 0.0145, -1.0] (7 numbers)

all checks passed: lengths match, 19 inputs / 7 targets per step, no NaN


In [16]:
# View one video as a table (inputs and targets side by side)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", "{:.4f}".format)

v = 0
df = pd.concat([pd.DataFrame(inputs[v][1],  columns=input_keys),
                pd.DataFrame(targets[v][1], columns=target_keys)], axis=1)
df.insert(0, "t", inputs[v][0])
df.head(10)

,t,eef_x,eef_y,eef_z,eef_qx,eef_qy,eef_qz,eef_qw,finger_1,finger_2,cube_x,cube_y,cube_z,cube_qx,cube_qy,cube_qz,cube_qw,grip2cube_x,grip2cube_y,grip2cube_z,act_dx,act_dy,act_dz,act_drx,act_dry,act_drz,act_grip
0,1,-0.0905,-0.0152,1.0118,0.9972,-0.0072,0.0740,0.0019,0.0208,-0.0208,0.0264,0.0270,0.8314,0.0000,0.0000,0.9691,0.2466,0.1169,0.0422,-0.1804,-0.0000,0.0000,0.0000,0.0038,0.1482,0.0145,-1.0000
1,2,-0.0940,-0.0130,1.0103,0.9977,-0.0052,0.0683,0.0012,0.0211,-0.0211,0.0264,0.0270,0.8193,0.0000,0.0000,0.9691,0.2466,0.1204,0.0400,-0.1910,0.2040,0.0870,-0.0720,0.0041,0.1379,0.0047,-1.0000
2,3,-0.0957,-0.0113,1.0085,0.9981,-0.0037,0.0620,0.0008,0.0231,-0.0231,0.0264,0.0270,0.8188,0.0000,0.0000,0.9691,0.2466,0.1222,0.0383,-0.1897,0.3230,0.1310,-0.0730,0.0067,0.1285,-0.0083,-1.0000
3,4,-0.0944,-0.0100,1.0065,0.9984,-0.0034,0.0569,0.0004,0.0264,-0.0264,0.0264,0.0270,0.8201,0.0000,0.0000,0.9691,0.2466,0.1208,0.0370,-0.1864,0.4910,0.2100,-0.1900,0.0082,0.1226,-0.0253,-1.0000
4,5,-0.0909,-0.0082,1.0037,0.9987,-0.0038,0.0518,-0.0001,0.0304,-0.0304,0.0264,0.0270,0.8207,0.0000,0.0000,0.9691,0.2466,0.1173,0.0352,-0.1830,0.4650,0.3090,-0.3040,0.0057,0.1096,-0.0458,-1.0000
5,6,-0.0861,-0.0057,0.9998,0.9989,-0.0051,0.0463,-0.0008,0.0341,-0.0340,0.0264,0.0270,0.8210,0.0000,0.0000,0.9691,0.2466,0.1125,0.0327,-0.1788,0.6030,0.2780,-0.2580,-0.0027,0.1004,-0.0592,-1.0000
6,7,-0.0799,-0.0032,0.9956,0.9991,-0.0076,0.0414,-0.0011,0.0363,-0.0362,0.0264,0.0270,0.8211,0.0000,0.0000,0.9691,0.2466,0.1064,0.0302,-0.1745,0.5890,0.2810,-0.2760,-0.0030,0.0890,-0.0740,-1.0000
7,8,-0.0731,-0.0005,0.9914,0.9993,-0.0110,0.0369,-0.0010,0.0375,-0.0374,0.0264,0.0270,0.8211,0.0000,0.0000,0.9691,0.2466,0.0995,0.0275,-0.1703,0.6800,0.3360,-0.2630,-0.0122,0.0729,-0.0889,-1.0000
8,9,-0.0653,0.0025,0.9874,0.9993,-0.0153,0.0331,-0.0006,0.0382,-0.0381,0.0264,0.0270,0.8212,0.0000,0.0000,0.9691,0.2466,0.0918,0.0245,-0.1662,0.6190,0.3620,-0.2630,-0.0110,0.0625,-0.1101,-1.0000
9,10,-0.0573,0.0060,0.9835,0.9994,-0.0207,0.0295,-0.0000,0.0385,-0.0385,0.0264,0.0270,0.8212,0.0000,0.0000,0.9691,0.2466,0.0838,0.0210,-0.1623,0.6150,0.2310,-0.2300,-0.0061,0.0594,-0.1153,-1.0000


## 5. How to use it later
```python
for (t, x), (_, y) in zip(inputs, targets):   # one video at a time
    x = np.array(x)   # (T_i, 19)
    y = np.array(y)   # (T_i, 7)
    # reset reservoir -> run over x -> collect features -> pair with y
```

In [18]:
import pickle
OUT = os.path.join(os.path.dirname(DATASET), "final_dataset.pkl")
with open(OUT, "wb") as fh:
    pickle.dump({"inputs": inputs, "targets": targets, "input_keys": input_keys,
                 "target_keys": target_keys, "demo_names": selected}, fh)
print("saved to", OUT, "| load with: pickle.load(open(path, 'rb'))")

saved to final_dataset.pkl | load with: pickle.load(open(path, 'rb'))
